# Lesson 1 - Semantic Search

Welcome to Lesson 1. 

To access the `requirement.txt` file, go to `File` and click on `Open`.
 
I hope you enjoy this course!

### Import the Needed Packages

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
import sys
print(sys.executable)
print(sys.version)

/Users/sandeep/Desktop/Projects/Vector-Databases/.venv/bin/python
3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.4.4.1)]


In [8]:
import json
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
from DLAIUtils import Utils
import torch

In [9]:
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

### Load the Dataset

In [10]:
dataset = load_dataset('quora', split='train[240000:290000]')

In [11]:
print( json.dumps(dataset[:5],indent = 4) )

{
    "questions": [
        {
            "id": [
                207550,
                351729
            ],
            "text": [
                "What is the truth of life?",
                "What's the evil truth of life?"
            ]
        },
        {
            "id": [
                33183,
                351730
            ],
            "text": [
                "Which is the best smartphone under 20K in India?",
                "Which is the best smartphone with in 20k in India?"
            ]
        },
        {
            "id": [
                351731,
                351732
            ],
            "text": [
                "Steps taken by Canadian government to improve literacy rate?",
                "Can I send homemade herbal hair oil from India to US via postal or private courier services?"
            ]
        },
        {
            "id": [
                37799,
                94186
            ],
            "text": [
                "What is a g

In [12]:
questions = []
for record in dataset['questions']:
    questions.extend(record['text'])
question = list(set(questions))
print('\n'.join(questions[:10]))
print('-' * 50)
print(f'Number of questions: {len(questions)}')

What is the truth of life?
What's the evil truth of life?
Which is the best smartphone under 20K in India?
Which is the best smartphone with in 20k in India?
Steps taken by Canadian government to improve literacy rate?
Can I send homemade herbal hair oil from India to US via postal or private courier services?
What is a good way to lose 30 pounds in 2 months?
What can I do to lose 30 pounds in 2 months?
Which of the following most accurately describes the translation of the graph y = (x+3)^2 -2 to the graph of y = (x -2)^2 +2?
How do you graph x + 2y = -2?
--------------------------------------------------
Number of questions: 100000


### Check cuda and Setup the model

**Note**: "Checking cuda" refers to checking if you have access to GPUs (faster compute). In this course, we are using CPUs. So, you might notice some code cells taking a little longer to run.

We are using *all-MiniLM-L6-v2* sentence-transformers model that maps sentences to a 384 dimensional dense vector space.

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    print('Sorry no cuda.')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

Sorry no cuda.


In [14]:
query = 'which city is the most populated in the world?'
xq = model.encode(query)
xq.shape

(384,)

### Setup Pinecone

In [15]:
utils = Utils()
print("Pinecone API Key:")
PINECONE_API_KEY = utils.get_pinecone_api_key()
print(PINECONE_API_KEY)

Pinecone API Key:
pcsk_Gfv9g_Nu5jnmaDS1bzw7ZbTon9wZnYjSUy4YTMFY3RAVfJzNMrBt2jX8PoLDgAPsuxoGK


In [16]:
pinecone = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = utils.create_dlai_index_name('dl-ai')

print(pinecone.list_indexes())

if INDEX_NAME in [index.name for index in pinecone.list_indexes()]:
    pinecone.delete_index(INDEX_NAME)
print(INDEX_NAME)
pinecone.create_index(name=INDEX_NAME, 
    dimension=model.get_sentence_embedding_dimension(), 
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region='us-east-1'))

index = pinecone.Index(INDEX_NAME)
print(index)

[{
    "name": "dl-ai-eyznhj0-fm00ios1ashzvewr2zwwyusye-aa",
    "metric": "cosine",
    "host": "dl-ai-eyznhj0-fm00ios1ashzvewr2zwwyusye-aa-f7p5236.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]
dl-ai-meyznhj0-fm00ios1ashzvewr2zwwyusye-a


### Create Embeddings and Upsert to Pinecone

In [17]:
batch_size=200
vector_limit=10000

questions = question[:vector_limit]

import json

for i in tqdm(range(0, len(questions), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(questions))
    # create IDs batch
    ids = [str(x) for x in range(i, i_end)]
    # create metadata batch
    metadatas = [{'text': text} for text in questions[i:i_end]]
    # create embeddings
    xc = model.encode(questions[i:i_end])
    # create records list for upsert
    records = zip(ids, xc, metadatas)
    # upsert to Pinecone
    index.upsert(vectors=records)

 66%|██████▌   | 33/50 [05:48<02:59, 10.55s/it]


UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 14 Apr 2026 07:17:57 GMT', 'Content-Type': 'text/plain', 'Content-Length': '12', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: Unauthorized


In [ ]:
index.describe_index_stats()

### Run Your Query

In [38]:
# small helper function so we can repeat queries later
def run_query(query):
  embedding = model.encode(query).tolist()
  results = index.query(top_k=10, vector=embedding, include_metadata=True, include_values=False)
  print(type(results))
  for result in results['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['text']}") 

In [ ]:
run_query('what to choose mtech vs psu for computer science')

In [ ]:
query = 'how do i make chocolate nice cake?'
run_query(query)